In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project = "openaire-graph")

In [ ]:
countries = ['FR', 'ES', 'PL','HR', 'DE', 'IT', 'PT', 'CZ', 'SI', 'NL']
countries_sql = ", ".join(f"'{c}'" for c in countries)

In [ ]:
eu_country_iso2 = [ "AT","BE", "BG", "HR", "CY", "CZ", "DK",
    "EE", "FI", "FR", "DE", "GR", "HU", "IE", "IT", "LV", "LT",
    "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE",
]
european_countries = ", ".join(f"'{c}'" for c in eu_country_iso2)

# Top 10 countries per number of DOA Publications produced by researcher's country

In [ ]:
query = f"""
with pubs as (
  select p.*, o.countryCode as orgCountry
  from `cnr_internal.european_doa_relations` r
  join cnr_internal.european_doa_publications p
  on p.id = r.source
  join `cnr_internal.european_doa_organizations` o
  on r.target = o.id
  where relationName = 'hasAuthorInstitution'
)
select orgCountry, count(distinct id) as n_publications
from pubs
where extract(year FROM DATE(publicationDate)) >= 2000 and
extract(year FROM DATE(publicationDate)) <= 2024
and orgCountry in ({european_countries})
group by orgCountry
order by n_publications desc
limit 50
"""
diamond_pubs_by_org_country = client.query(query).to_dataframe()

In [ ]:
diamond_pubs_by_org_country

# Number of products received by journals in top 10 countries

In [ ]:
query = f"""
with issn_match as (
select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join `cnr_internal.european_doa_publications` p
on d.issnPrinted = p.issnPrinted and d.issnPrinted is not null
),
eissn_match as (
  select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join `cnr_internal.european_doa_publications` p
on d.issnOnline = p.issnOnline and d.issnOnline is not null
),
all_pubs as (
select * from issn_match
union distinct
select * from eissn_match
)
select publisherCountry, count(distinct id) as nProducts
from all_pubs
where publisherCountry is not null and extract(year from date(publicationDate)) >= 2000 and extract(year from date(publicationDate)) <= 2024
and publisherCountry in ({countries_sql})
group by publisherCountry
order by nProducts desc
"""
doa_countries_stats_by_journal_country = client.query(query).to_dataframe()

In [ ]:
doa_countries_stats_by_journal_country

# Article-side production for the top 10 Countries

## From 2000-2024

In [ ]:
query = f"""
with pubs as (
  select p.id,p.publicationDate, p.issnLinking, p.issnPrinted, p.issnOnline, o.countryCode as orgCountry
  from oag_v10_6_0.relations r
  join oag_v10_6_0.publications p
  on p.id = r.source
  join oag_v10_6_0.organizations o
  on r.target = o.id
  where relationName = 'hasAuthorInstitution'
)
select orgCountry, count(distinct id) as n_publications
from pubs
where extract(year FROM DATE(publicationDate)) >= 2000 and
extract(year FROM DATE(publicationDate)) <= 2024
and orgCountry in ({countries_sql}) and (issnLinking is not null or issnPrinted is not null or issnOnline is not null)
group by orgCountry
order by n_publications desc
"""
total_publications = client.query(query).to_dataframe()

In [ ]:
total_publications

## From 2020-2024

In [ ]:
query = f"""
with pubs as (
  select p.id,p.publicationDate, p.issnLinking, p.issnPrinted, p.issnOnline, o.countryCode as orgCountry
  from oag_v10_6_0.relations r
  join oag_v10_6_0.publications p
  on p.id = r.source
  join oag_v10_6_0.organizations o
  on r.target = o.id
  where relationName = 'hasAuthorInstitution'
)
select orgCountry, count(distinct id) as n_publications
from pubs
where extract(year FROM DATE(publicationDate)) >= 2020 and
extract(year FROM DATE(publicationDate)) <= 2024
and orgCountry in ({countries_sql}) and (issnLinking is not null or issnPrinted is not null or issnOnline is not null)
group by orgCountry
order by n_publications desc
"""
total_publications_5 = client.query(query).to_dataframe()

In [ ]:
total_publications_5

# Journal-side received articles for the top 10 countries

In [ ]:
query = f"""
with issn_match as (
select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join oag_v10_6_0.publications p
on d.issnPrinted = p.issnPrinted and d.issnPrinted is not null
),
eissn_match as (
  select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join oag_v10_6_0.publications p
on d.issnOnline = p.issnOnline and d.issnOnline is not null
),
all_pubs as (
select * from issn_match
union distinct
select * from eissn_match
)
select publisherCountry, count(distinct jourid) as n_journals, count(distinct id) as nProducts
from all_pubs
where publisherCountry is not null
and extract(year from date(publicationDate)) >= 2000 and extract(year from date(publicationDate)) <= 2024
and publisherCountry in ({countries_sql})
group by publisherCountry
order by n_journals desc
"""
total_journals_production = client.query(query).to_dataframe()

In [ ]:
total_journals_production

# Year Distributions

### # of active DOA Journals per Year

In [ ]:
query = f"""
with issn_match as (
select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join oag_v10_6_0.publications p
on d.issnPrinted = p.issnPrinted and d.issnPrinted is not null
),
eissn_match as (
  select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join oag_v10_6_0.publications p
on d.issnOnline = p.issnOnline and d.issnOnline is not null
),
all_pubs as (
select * from issn_match
union distinct
select * from eissn_match
)
select distinct extract(year FROM DATE(publicationDate)) as year, count(distinct jourid) as n_journals
from all_pubs
where extract(year FROM DATE(publicationDate)) >= 2000 and
extract(year FROM DATE(publicationDate)) <= 2024
and publisherCountry in ({european_countries})
group by year
order by year asc
"""
journals_x_year_df = client.query(query).to_dataframe()

### # of DOA articles per year

In [ ]:
query = f"""
with pubs as (
  select p.*, o.countryCode as orgCountry
  from `cnr_internal.european_doa_relations` r
  join cnr_internal.european_doa_publications p
  on p.id = r.source
  join `cnr_internal.european_doa_organizations` o
  on r.target = o.id
  where relationName = 'hasAuthorInstitution'
)
select distinct extract(year FROM DATE(publicationDate)) as year, count(distinct id) as n_publications
from pubs
where extract(year FROM DATE(publicationDate)) >= 2000 and
extract(year FROM DATE(publicationDate)) <= 2024
and orgCountry in ({european_countries})
group by year
order by year asc
"""
pubs_x_year_df = client.query(query).to_dataframe()

## Article-Side Analysis

### # of DOA journals for the top 10 countries per year

In [ ]:
query = f"""
with issn_match as (
select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join `cnr_internal.european_doa_publications` p
on d.issnPrinted = p.issnPrinted and d.issnPrinted is not null
),
eissn_match as (
  select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join `cnr_internal.european_doa_publications` p
on d.issnOnline = p.issnOnline and d.issnOnline is not null
),
all_pubs as (
select * from issn_match
union distinct
select * from eissn_match
)
select distinct extract(year FROM DATE(publicationDate)) as year, publisherCountry, count(distinct jourid) as n_journals
from all_pubs
where extract(year FROM DATE(publicationDate)) >= 2000 and
extract(year FROM DATE(publicationDate)) <= 2024 and publisherCountry is not null
and publisherCountry in ({countries_sql})
group by year, publisherCountry
order by year asc
"""
journals_x_country_year_df = client.query(query).to_dataframe()

### # of DOA Articles for the top 10 countries per year

In [ ]:
query = f"""
with pubs as (
  select p.*, o.countryCode as orgCountry
  from `cnr_internal.european_doa_relations` r
  join cnr_internal.european_doa_publications p
  on p.id = r.source
  join `cnr_internal.european_doa_organizations` o
  on r.target = o.id
  where relationName = 'hasAuthorInstitution'
)
select distinct extract(year FROM DATE(publicationDate)) as year, orgCountry, count(distinct id) as Articles
from pubs
where extract(year FROM DATE(publicationDate)) >= 2000 and
extract(year FROM DATE(publicationDate)) <= 2024
and orgCountry in ({countries_sql})
group by year, orgCountry
order by year asc
"""
pubs_x_country_year_df = client.query(query).to_dataframe()

## Journal-side Analysis

### # of DOA Articles received for the top 10 Countries

In [ ]:
query = f"""
with issn_match as (
select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join `cnr_internal.european_doa_publications` p
on d.issnPrinted = p.issnPrinted and d.issnPrinted is not null
),
eissn_match as (
  select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join `cnr_internal.european_doa_publications` p
on d.issnOnline = p.issnOnline and d.issnOnline is not null
),
all_pubs as (
select * from issn_match
union distinct
select * from eissn_match
)
select distinct extract(year FROM DATE(publicationDate)) as year, publisherCountry, count(distinct id) as nProducts
from all_pubs
where extract(year FROM DATE(publicationDate)) >= 2000 and
extract(year FROM DATE(publicationDate)) <= 2024 and publisherCountry is not null
and publisherCountry in ({countries_sql})
group by year, publisherCountry
order by year asc
"""
journals_article_x_country_year_df = client.query(query).to_dataframe()

# International Appeal

In [ ]:
%%bigquery
with issn_match as (
select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join `cnr_internal.european_doa_publications` p
on d.issnPrinted = p.issnPrinted and d.issnPrinted is not null
),
eissn_match as (
  select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from cnr_internal.european_doa_datasources d
join `cnr_internal.european_doa_publications` p
on d.issnOnline = p.issnOnline and d.issnOnline is not null
),
all_pubs as (
select * from issn_match
union distinct
select * from eissn_match
),
pubs as (
  select p.id, p.publicationDate, p.publisherCountry, o.countryCode as orgCountry
  from `cnr_internal.european_doa_relations` r
  join all_pubs p
  on p.id = r.source
  join `cnr_internal.european_doa_organizations` o
  on r.target = o.id
  where relationName = 'hasAuthorInstitution' and extract(year from date(p.publicationDate)) >= 2000 and extract(year from date(p.publicationDate)) <= 2024
),
pubs_all_foreign as (
  select id,publisherCountry
  from pubs
  group by id, publisherCountry
  having countif(orgCountry = publisherCountry) = 0
)
select publisherCountry, count(*) as n_publications
from pubs_all_foreign
group by publisherCountry
order by publisherCountry asc

# Citation Analysis

## European Overview

In [ ]:
query = f"""
with pubs as (
  select p.id,p.publicationDate, p.instances, p.subjects, o.countryCode as orgCountry
  from cnr_internal.european_doa_relations r
  join cnr_internal.european_doa_publications p
  on p.id = r.source
  join `oag_v10_6_0.organizations` o
  on r.target = o.id
  where relationName = 'hasAuthorInstitution'
),
citations as (
  select p.*, r.source as citingId
  from oag_v10_6_0.relations r
  join pubs p
  on p.id = r.target
  where r.relationName = 'Cites'
),
citing as (
  select c.*, extract(year from date(p.publicationDate)) as citationYear
  from citations c
  join oag_v10_6_0.publications p
  on c.citingId = p.id
)
select citationYear as year, count(citingId) as CitationsCount
from citing
where extract(year from date(publicationDate)) >= 2000 and extract(year from date(publicationDate)) <= 2024 and citationYear >= 2000 and citationYear <= 2024
and orgCountry in ({european_countries})
group by year
"""
all_citations_received_by_researcher_country_citing_year = client.query(query).to_dataframe()

## Citation received by researchers affiliated with an organization in the Top 10 Countries (Article Side)

### According to the publication date of the cited article

In [ ]:
query = f"""
with pubs as (
  select p.id,p.publicationDate, p.instances, p.subjects, o.countryCode as orgCountry
  from cnr_internal.european_doa_relations r
  join cnr_internal.european_doa_publications p
  on p.id = r.source
  join `oag_v10_6_0.organizations` o
  on r.target = o.id
  where relationName = 'hasAuthorInstitution'
),
citations as (
  select p.*, r.source as citingId
  from oag_v10_6_0.relations r
  join pubs p
  on p.id = r.target
  where r.relationName = 'Cites'
),
citing as (
  select c.*, extract(year from date(p.publicationDate)) as citationYear
  from citations c
  join oag_v10_6_0.publications p
  on c.citingId = p.id
)
select orgCountry, extract(year from date(publicationDate)) as year, count(citingId) as CitationsCount
from citing
where orgCountry in ({countries_sql})
and extract(year from date(publicationDate)) >= 2000 and extract(year from date(publicationDate)) <= 2024
group by orgCountry, year
"""
citations = client.query(query).to_dataframe()

### According to the publication date of the article citing diamond

In [ ]:
query = f"""
with pubs as (
  select p.id,p.publicationDate, p.instances, p.subjects, o.countryCode as orgCountry
  from cnr_internal.european_doa_relations r
  join cnr_internal.european_doa_publications p
  on p.id = r.source
  join `oag_v10_6_0.organizations` o
  on r.target = o.id
  where relationName = 'hasAuthorInstitution'
),
citations as (
  select p.*, r.source as citingId
  from oag_v10_6_0.relations r
  join pubs p
  on p.id = r.target
  where r.relationName = 'Cites'
),
citing as (
  select c.*, extract(year from date(p.publicationDate)) as citationYear
  from citations c
  join oag_v10_6_0.publications p
  on c.citingId = p.id
)
select orgCountry, citationYear, count(citingId) as CitationsCount
from citing
where orgCountry in ({countries_sql})
and extract(year from date(publicationDate)) >= 2000 and extract(year from date(publicationDate)) <= 2024 and citationYear >= 2000 and citationYear <= 2024
group by orgCountry, citationYear
"""
citations_1 = client.query(query).to_dataframe()

## Citations received by articles submitted DOA Journals in the Top 10 Countries (Journal-Side)

In [ ]:
query = f"""
with datasources as (
  select d.*, subject
  from cnr_internal.european_doa_datasources d
  join cnr_internal.road r
  on (d.issnPrinted = r.issn and d.issnPrinted is not null) or (d.issnOnline = r.issn and d.issnOnline is not null),
  unnest(r.fos) as subject
),
issn_match as (
select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from datasources d
join `cnr_internal.european_doa_publications` p
on d.issnPrinted = p.issnPrinted and d.issnPrinted is not null
),
eissn_match as (
  select p.id, p.publicationDate, d.id as jourid, d.publisherCountry
from datasources d
join `cnr_internal.european_doa_publications` p
on d.issnOnline = p.issnOnline and d.issnOnline is not null
),
all_pubs as (
select * from issn_match
union distinct
select * from eissn_match
),
citations as (
  select p.*, r.source as citingId
  from oag_v10_6_0.relations r
  join all_pubs p
  on p.id = r.target
  where r.relationName = 'Cites'
)
select publisherCountry, extract(year from date(publicationDate)) as year, count(citingId) as CitationsCount
from citations
where extract(year from date(publicationDate)) >= 2000 and extract(year from date(publicationDate)) <= 2024
and publisherCountry in ({european_countries})
group by publisherCountry, year
"""
citations_by_publisher_org = client.query(query).to_dataframe()

# Discipline Coverage per top 10 Countries

## Distribution of DOA Journals per Discipline in Europe.

In [ ]:
query = """
with datasources as (
  select d.*, subject
  from cnr_internal.european_doa_datasources d
  join cnr_internal.road r
  on (d.issnPrinted = r.issn and d.issnPrinted is not null) or (d.issnOnline = r.issn and d.issnOnline is not null),
  unnest(r.fos) as subject
),
issn_match as (
select p.id, p.publicationDate, d.id as jourid, d.publisherCountry, d.subject
from datasources d
join `cnr_internal.european_doa_publications` p
on d.issnPrinted = p.issnPrinted and d.issnPrinted is not null
),
eissn_match as (
  select p.id, p.publicationDate, d.id as jourid, d.publisherCountry, d.subject
from datasources d
join `cnr_internal.european_doa_publications` p
on d.issnOnline = p.issnOnline and d.issnOnline is not null
),
all_pubs as (
select * from issn_match
union distinct
select * from eissn_match
)
select distinct extract(year FROM DATE(publicationDate)) as year, subject as subjName, count(distinct jourid) as n_journals
from all_pubs
where extract(year FROM DATE(publicationDate)) >= 2000 and
extract(year FROM DATE(publicationDate)) <= 2024
group by year, subjName
order by year asc
"""
journals_x_discipline = client.query(query).to_dataframe()

## Journal-Side

In [ ]:
query = f"""
with datasources as (
  select d.*, subject
  from cnr_internal.european_doa_datasources d
  join cnr_internal.road r
  on (d.issnPrinted = r.issn and d.issnPrinted is not null) or (d.issnOnline = r.issn and d.issnOnline is not null),
  unnest(r.fos) as subject
),
issn_match as (
select p.id, p.publicationDate, d.id as jourid, d.publisherCountry, d.subject
from datasources d
join `cnr_internal.european_doa_publications` p
on d.issnPrinted = p.issnPrinted and d.issnPrinted is not null
),
eissn_match as (
  select p.id, p.publicationDate, d.id as jourid, d.publisherCountry, d.subject
from datasources d
join `cnr_internal.european_doa_publications` p
on d.issnOnline = p.issnOnline and d.issnOnline is not null
),
all_pubs as (
select * from issn_match
union distinct
select * from eissn_match
)
select subject, publisherCountry, count(distinct id) as nProducts
from all_pubs
where extract(year from date(publicationDate)) >= 2000 and extract(year from date(publicationDate)) <= 2024
and publisherCountry in ({countries_sql})
group by subject, publisherCountry
order by subject asc
"""
journals_discipline_x_country = client.query(query).to_dataframe()

## Article-Side

In [ ]:
query = f"""
with datasources as (
  select d.*, subject
  from cnr_internal.european_doa_datasources d
  join cnr_internal.road r
  on (d.issnPrinted = r.issn and d.issnPrinted is not null) or (d.issnOnline = r.issn and d.issnOnline is not null),
  unnest(r.fos) as subject
),
issn_match as (
select p.id, p.publicationDate, d.id as jourid, d.publisherCountry, d.subject
from datasources d
join `cnr_internal.european_doa_publications` p
on d.issnPrinted = p.issnPrinted and d.issnPrinted is not null
),
eissn_match as (
  select p.id, p.publicationDate, d.id as jourid, d.publisherCountry, d.subject
from datasources d
join `cnr_internal.european_doa_publications` p
on d.issnOnline = p.issnOnline and d.issnOnline is not null
),
all_pubs as (
select * from issn_match
union distinct
select * from eissn_match
),
pubs as (
  select p.id, p.publicationDate, o.countryCode, p.subject
from `cnr_internal.european_doa_relations` r
join `cnr_internal.european_doa_organizations` o
on r.target = o.id
join all_pubs p
on r.source = p.id
where relationName = 'hasAuthorInstitution'
)
select subject, countryCode, count(distinct id) as nArticles
from pubs
where extract(year from date(publicationDate)) >= 2000 and extract(year from date(publicationDate)) <= 2024
and countryCode in ({countries_sql})
group by subject, countryCode
order by subject, nArticles desc
"""
disciplines = client.query(query).to_dataframe()